
# Notebook 23 — Universality Boundary Detection and Phase Transition Surfaces

This notebook extends Notebook 22 from OOD transfer into controlled boundary detection.

Core question:

> Where do graph families stop behaving like one residual universality class and transition into another?

Boundary signals:
- nearest-family switches,
- low relative-confidence regions,
- high-curvature trajectories,
- distance minima between competing known universality branches.


In [ ]:

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

np.random.seed(42)

def detect_repo_root():
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    if (cwd / "notebooks").exists() or (cwd / ".git").exists():
        return cwd
    if Path("/content").exists():
        for c in Path("/content").glob("*"):
            if c.is_dir() and ((c / "notebooks").exists() or (c / ".git").exists()):
                return c
        return Path("/content")
    return cwd

ROOT = detect_repo_root()
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
DOCS = ROOT / "docs"

for folder in [RESULTS, FIGURES, DOCS]:
    folder.mkdir(exist_ok=True)

print("cwd:", Path.cwd())
print("repo root:", ROOT)
print("results:", RESULTS)
print("figures:", FIGURES)


In [ ]:

GRAPH_SIZES = [16, 32, 64, 128]
BASE_N = 64

def safe_connected_largest_component(G):
    G = nx.Graph(G)
    G.remove_edges_from(nx.selfloop_edges(G))
    if len(G) == 0:
        return G
    if nx.is_connected(G):
        return G
    largest = max(nx.connected_components(G), key=len)
    return G.subgraph(largest).copy()

def ring_lattice_graph(n):
    return nx.watts_strogatz_graph(n, 4, 0.0, seed=100 + n)

def small_world_graph(n):
    return nx.watts_strogatz_graph(n, 4, 0.25, seed=200 + n)

def er_graph(n):
    G = nx.erdos_renyi_graph(n, 0.10, seed=300 + n)
    return safe_connected_largest_component(G)

def scale_free_graph(n):
    return nx.barabasi_albert_graph(n, 3, seed=400 + n)

def modular_clustered_graph(n):
    sizes = [n // 2, n - n // 2]
    probs = [[0.18, 0.012], [0.012, 0.18]]
    G = nx.stochastic_block_model(sizes, probs, seed=500 + n)
    return safe_connected_largest_component(G)

BASE_GENERATORS = {
    "ring lattice": ring_lattice_graph,
    "small world": small_world_graph,
    "Erdos-Renyi": er_graph,
    "scale free": scale_free_graph,
    "modular clustered": modular_clustered_graph,
}


## 1. Residual feature extraction

In [ ]:

def normalized_entropy(weights):
    weights = np.asarray(weights, dtype=float)
    weights = np.abs(weights)
    total = weights.sum()
    if total <= 0:
        return 0.0
    p = weights / total
    p = p[p > 0]
    return float(-sum(float(pi) * np.log(float(pi)) for pi in p) / np.log(len(weights)))

def safe_average_path_length(G):
    G = safe_connected_largest_component(G)
    if len(G) <= 1:
        return 0.0
    try:
        return float(nx.average_shortest_path_length(G))
    except Exception:
        return 0.0

def graph_features(G):
    G = safe_connected_largest_component(G)
    n_nodes = max(len(G.nodes), 1)
    n_edges = len(G.edges)

    A = nx.to_numpy_array(G)
    eigvals = np.linalg.eigvalsh(A) if A.size else np.array([0.0])
    eigvals = np.sort(np.abs(eigvals))[::-1]

    degree = np.array([d for _, d in G.degree()], dtype=float)
    if len(degree) == 0:
        degree = np.array([0.0])

    clustering_values = list(nx.clustering(G).values()) if len(G) else [0.0]
    clustering = float(np.mean(clustering_values)) if clustering_values else 0.0
    path_length = safe_average_path_length(G)

    residual = eigvals - np.mean(eigvals)
    abs_residual = np.abs(residual)

    mean_abs = float(abs_residual.mean()) if len(abs_residual) else 0.0
    max_abs = float(abs_residual.max()) if len(abs_residual) else 0.0

    bend_energy = float(np.sum(np.diff(residual, n=2) ** 2)) if len(residual) > 2 else 0.0
    curvature = float(np.sum(np.abs(np.diff(residual)))) if len(residual) > 1 else 0.0
    localization = max_abs / (mean_abs + 1e-9)

    spectral_gap = float(eigvals[0] - eigvals[1]) if len(eigvals) > 1 else float(eigvals[0])
    spectral_ratio = float(eigvals[1] / (eigvals[0] + 1e-9)) if len(eigvals) > 1 else 0.0
    density = float(2 * n_edges / max(n_nodes * (n_nodes - 1), 1))

    return {
        "mean_abs_residual": mean_abs,
        "max_abs_residual": max_abs,
        "total_residual_energy": float(np.sum(residual ** 2)),
        "residual_localization": float(localization),
        "residual_entropy": normalized_entropy(abs_residual),
        "residual_curvature": curvature,
        "residual_bend_energy": bend_energy,
        "mean_degree": float(np.mean(degree)),
        "degree_std": float(np.std(degree)),
        "clustering": clustering,
        "path_length": path_length,
        "spectral_gap": spectral_gap,
        "spectral_ratio": spectral_ratio,
        "density": density,
        "n_connected_nodes": int(n_nodes),
        "n_edges": int(n_edges),
    }


## 2. Build baseline residual manifold

In [ ]:

baseline_rows = []

for topology, generator in BASE_GENERATORS.items():
    for N in GRAPH_SIZES:
        G = generator(N)
        feats = graph_features(G)
        feats["topology"] = topology
        feats["N"] = N
        feats["source"] = "baseline"
        baseline_rows.append(feats)

baseline_df = pd.DataFrame(baseline_rows)

feature_cols = [
    c for c in baseline_df.columns
    if c not in ["topology", "N", "source"]
]

X = baseline_df[feature_cols].replace([np.inf, -np.inf], np.nan)

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
pca = PCA(n_components=2)

X_imp = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imp)
coords = pca.fit_transform(X_scaled)

baseline_df["PC1"] = coords[:, 0]
baseline_df["PC2"] = coords[:, 1]

baseline_df.to_csv(RESULTS / "boundary_baseline_residual_manifold.csv", index=False)

centroids = baseline_df.groupby("topology")[["PC1", "PC2"]].mean()
fixed_points = baseline_df.sort_values("N").groupby("topology")[["PC1", "PC2"]].last()

print("feature columns:", feature_cols)
print("PCA variance:", pca.explained_variance_ratio_)
baseline_df.head()


## 3. Controlled boundary sweep generators

In [ ]:

def ws_rewiring_boundary(n=BASE_N, beta=0.0, seed=0):
    return nx.watts_strogatz_graph(n, 4, float(beta), seed=seed)

def modular_bridge_boundary(n=BASE_N, p_out=0.01, p_in=0.18, seed=0):
    sizes = [n // 2, n - n // 2]
    probs = [[p_in, float(p_out)], [float(p_out), p_in]]
    G = nx.stochastic_block_model(sizes, probs, seed=seed)
    return safe_connected_largest_component(G)

def degree_heterogeneity_boundary(n=BASE_N, m=1, seed=0):
    m = int(max(1, min(m, n - 1)))
    return nx.barabasi_albert_graph(n, m, seed=seed)

def chord_density_boundary(n=BASE_N, chord_density=0.0, seed=0):
    rng = np.random.default_rng(seed)
    G = nx.cycle_graph(n)

    possible = []
    for i in range(n):
        for j in range(i + 2, n):
            if (i == 0 and j == n - 1):
                continue
            possible.append((i, j))

    n_chords = int(round(float(chord_density) * len(possible)))
    if n_chords > 0:
        idx = rng.choice(len(possible), size=min(n_chords, len(possible)), replace=False)
        for k in idx:
            G.add_edge(*possible[int(k)])
    return G

SWEEPS = {
    "ws_rewiring": {
        "parameter_name": "beta",
        "values": np.linspace(0.0, 1.0, 25),
        "generator": lambda value, seed: ws_rewiring_boundary(beta=value, seed=seed),
        "label": "Watts-Strogatz rewiring",
    },
    "modular_bridge": {
        "parameter_name": "p_out",
        "values": np.linspace(0.002, 0.08, 25),
        "generator": lambda value, seed: modular_bridge_boundary(p_out=value, seed=seed),
        "label": "SBM modular bridge",
    },
    "degree_heterogeneity": {
        "parameter_name": "m",
        "values": np.array([1, 2, 3, 4, 5, 6, 8, 10, 12]),
        "generator": lambda value, seed: degree_heterogeneity_boundary(m=value, seed=seed),
        "label": "Preferential attachment degree heterogeneity",
    },
    "chord_density": {
        "parameter_name": "chord_density",
        "values": np.linspace(0.0, 0.25, 25),
        "generator": lambda value, seed: chord_density_boundary(chord_density=value, seed=seed),
        "label": "Cycle chord density",
    },
}


## 4. Project boundary sweeps into baseline manifold

In [ ]:

def project_features(df):
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X_imp = imputer.transform(X)
    X_scaled = scaler.transform(X_imp)
    coords = pca.transform(X_scaled)
    out = df.copy()
    out["PC1"] = coords[:, 0]
    out["PC2"] = coords[:, 1]
    return out

def nearest_family(pc1, pc2):
    vec = np.array([pc1, pc2], dtype=float)
    distances = {
        topology: float(np.linalg.norm(vec - centroids.loc[topology].values))
        for topology in centroids.index
    }
    ordered = sorted(distances.items(), key=lambda item: item[1])
    nearest = ordered[0]
    second = ordered[1]
    return {
        "nearest_family": nearest[0],
        "nearest_distance": nearest[1],
        "second_family": second[0],
        "second_distance": second[1],
        "confidence_gap": second[1] - nearest[1],
        "relative_confidence": (second[1] - nearest[1]) / max(second[1], 1e-9),
    }

sweep_rows = []

for sweep_name, spec in SWEEPS.items():
    for idx, value in enumerate(spec["values"]):
        G = spec["generator"](value, seed=12000 + idx)
        feats = graph_features(G)
        feats["sweep"] = sweep_name
        feats["sweep_label"] = spec["label"]
        feats["parameter_name"] = spec["parameter_name"]
        feats["parameter_value"] = float(value)
        feats["N"] = BASE_N
        sweep_rows.append(feats)

sweep_df = pd.DataFrame(sweep_rows)
sweep_df = project_features(sweep_df)

assign_df = pd.DataFrame([
    nearest_family(row["PC1"], row["PC2"])
    for _, row in sweep_df.iterrows()
])

sweep_df = pd.concat([sweep_df.reset_index(drop=True), assign_df], axis=1)

sweep_df.to_csv(RESULTS / "boundary_all_sweeps.csv", index=False)

for sweep_name in SWEEPS:
    sweep_df[sweep_df["sweep"] == sweep_name].to_csv(
        RESULTS / f"boundary_{sweep_name}.csv",
        index=False,
    )

sweep_df.head()


## 5. Boundary switch and curvature detection

In [ ]:

def path_curvature(xs, ys):
    pts = np.column_stack([xs, ys])
    if len(pts) < 3:
        return np.zeros(len(pts))
    curvatures = np.zeros(len(pts))
    for i in range(1, len(pts) - 1):
        v1 = pts[i] - pts[i - 1]
        v2 = pts[i + 1] - pts[i]
        denom = np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-9
        cosang = np.clip(np.dot(v1, v2) / denom, -1, 1)
        curvatures[i] = np.arccos(cosang)
    return curvatures

switch_rows = []
curve_rows = []

for sweep_name, sub in sweep_df.groupby("sweep"):
    sub = sub.sort_values("parameter_value").reset_index(drop=True)
    fam = list(sub["nearest_family"])
    params = list(sub["parameter_value"])
    curv = path_curvature(sub["PC1"].values, sub["PC2"].values)

    for i, row in sub.iterrows():
        curve_rows.append({
            "sweep": sweep_name,
            "parameter_value": row["parameter_value"],
            "nearest_family": row["nearest_family"],
            "relative_confidence": row["relative_confidence"],
            "path_curvature": float(curv[i]),
        })

    switch_points = []
    for i in range(1, len(fam)):
        if fam[i] != fam[i - 1]:
            switch_points.append({
                "from_family": fam[i - 1],
                "to_family": fam[i],
                "parameter_left": params[i - 1],
                "parameter_right": params[i],
                "parameter_midpoint": 0.5 * (params[i - 1] + params[i]),
            })

    min_conf_pos = int(sub["relative_confidence"].to_numpy().argmin())
    max_curve_pos = int(np.argmax(curv))

    switch_rows.append({
        "sweep": sweep_name,
        "sweep_label": sub["sweep_label"].iloc[0],
        "parameter_name": sub["parameter_name"].iloc[0],
        "n_switches": len(switch_points),
        "switch_points": json.dumps(switch_points),
        "min_confidence_parameter": float(sub.iloc[min_conf_pos]["parameter_value"]),
        "min_relative_confidence": float(sub.iloc[min_conf_pos]["relative_confidence"]),
        "max_curvature_parameter": float(sub.iloc[max_curve_pos]["parameter_value"]),
        "max_path_curvature": float(curv[max_curve_pos]),
        "start_family": fam[0],
        "end_family": fam[-1],
    })

switch_df = pd.DataFrame(switch_rows)
curve_df = pd.DataFrame(curve_rows)

switch_df.to_csv(RESULTS / "boundary_switch_points.csv", index=False)
curve_df.to_csv(RESULTS / "boundary_curve_diagnostics.csv", index=False)

switch_df


## 6. Boundary paths

In [ ]:

def plot_boundary_path(sweep_name, filename):
    sub = sweep_df[sweep_df["sweep"] == sweep_name].sort_values("parameter_value")
    spec = SWEEPS[sweep_name]

    plt.figure(figsize=(10, 7))

    for topology in baseline_df["topology"].unique():
        base = baseline_df[baseline_df["topology"] == topology]
        plt.scatter(base["PC1"], base["PC2"], alpha=0.18, s=120)

    for topology, row in centroids.iterrows():
        plt.scatter(row["PC1"], row["PC2"], marker="*", s=240, color="black")
        plt.text(row["PC1"], row["PC2"], topology, fontsize=9, weight="bold")

    sc = plt.scatter(
        sub["PC1"], sub["PC2"],
        c=sub["parameter_value"],
        s=100,
        cmap="viridis",
        edgecolor="black",
        linewidth=0.3,
    )
    plt.plot(sub["PC1"], sub["PC2"], linewidth=2.0, alpha=0.8)

    for _, row in sub.iloc[::max(1, len(sub)//6)].iterrows():
        plt.text(row["PC1"], row["PC2"], f"{row['parameter_value']:.3g}", fontsize=8)

    plt.colorbar(sc, label=spec["parameter_name"])
    plt.axhline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.45)
    plt.axvline(0, linestyle="--", linewidth=0.8, color="black", alpha=0.45)
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    plt.title(spec["label"])
    plt.grid(alpha=0.3)

    path = FIGURES / filename
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    print("saved:", path)

plot_boundary_path("ws_rewiring", "boundary_ws_rewiring_path.png")
plot_boundary_path("modular_bridge", "boundary_modular_bridge_path.png")
plot_boundary_path("degree_heterogeneity", "boundary_degree_heterogeneity_path.png")
plot_boundary_path("chord_density", "boundary_chord_density_path.png")


## 7. Assignment and confidence curves

In [ ]:

family_to_int = {fam: i for i, fam in enumerate(centroids.index)}
int_to_family = {i: fam for fam, i in family_to_int.items()}

def plot_assignment_curve(sweep_name, filename):
    sub = sweep_df[sweep_df["sweep"] == sweep_name].sort_values("parameter_value")
    spec = SWEEPS[sweep_name]
    y = sub["nearest_family"].map(family_to_int)

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    axes[0].plot(sub["parameter_value"], y, marker="o", linewidth=2)
    axes[0].set_yticks(list(int_to_family.keys()))
    axes[0].set_yticklabels(list(int_to_family.values()))
    axes[0].set_ylabel("nearest family")
    axes[0].set_title(f"{spec['label']} — nearest-family assignment")
    axes[0].grid(alpha=0.3)

    axes[1].plot(sub["parameter_value"], sub["relative_confidence"], marker="o", linewidth=2)
    axes[1].set_xlabel(spec["parameter_name"])
    axes[1].set_ylabel("relative confidence")
    axes[1].set_title("boundary confidence")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    path = FIGURES / filename
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    print("saved:", path)

plot_assignment_curve("ws_rewiring", "boundary_ws_family_assignment.png")
plot_assignment_curve("modular_bridge", "boundary_modular_family_assignment.png")
plot_assignment_curve("degree_heterogeneity", "boundary_degree_family_assignment.png")
plot_assignment_curve("chord_density", "boundary_chord_family_assignment.png")


## 8. Boundary surface summary

In [ ]:

summary_surface = switch_df.copy()
summary_surface["transition_label"] = summary_surface["start_family"] + " -> " + summary_surface["end_family"]
summary_surface.to_csv(RESULTS / "boundary_surface_summary.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = summary_surface.sort_values("min_relative_confidence")
axes[0].barh(plot_df["sweep_label"], plot_df["min_relative_confidence"])
axes[0].set_xlabel("minimum relative confidence")
axes[0].set_title("Boundary confidence minima")
axes[0].grid(alpha=0.3, axis="x")

plot_df = summary_surface.sort_values("max_path_curvature")
axes[1].barh(plot_df["sweep_label"], plot_df["max_path_curvature"])
axes[1].set_xlabel("maximum path curvature")
axes[1].set_title("Boundary curvature maxima")
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
path = FIGURES / "boundary_surface_summary.png"
plt.savefig(path, dpi=220, bbox_inches="tight")
plt.show()
print("saved:", path)

summary_surface


## 9. Combined diagnostics

In [ ]:

plt.figure(figsize=(10, 6))
for sweep_name, sub in curve_df.groupby("sweep"):
    label = SWEEPS[sweep_name]["label"]
    plt.plot(sub["parameter_value"], sub["relative_confidence"], marker="o", linewidth=2, label=label)

plt.xlabel("sweep parameter value")
plt.ylabel("relative confidence")
plt.title("Boundary confidence minima across controlled sweeps")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
path = FIGURES / "boundary_confidence_minima.png"
plt.savefig(path, dpi=220, bbox_inches="tight")
plt.show()
print("saved:", path)

plt.figure(figsize=(10, 6))
for sweep_name, sub in curve_df.groupby("sweep"):
    label = SWEEPS[sweep_name]["label"]
    plt.plot(sub["parameter_value"], sub["path_curvature"], marker="o", linewidth=2, label=label)

plt.xlabel("sweep parameter value")
plt.ylabel("path curvature")
plt.title("Boundary curvature across controlled sweeps")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
path = FIGURES / "boundary_curvature_diagnostics.png"
plt.savefig(path, dpi=220, bbox_inches="tight")
plt.show()
print("saved:", path)


## 10. Export summary

In [ ]:

summary = {
    "notebook": "23_universality_boundary_detection.ipynb",
    "core_question": "Where do graph families stop behaving like one residual universality class and transition into another?",
    "core_claim": (
        "Controlled graph-family interpolations reveal residual universality boundaries "
        "as low-confidence regions, nearest-family switches, and high-curvature paths "
        "in residual manifold coordinates."
    ),
    "baseline_topologies": list(BASE_GENERATORS.keys()),
    "graph_sizes": GRAPH_SIZES,
    "base_N_for_boundary_sweeps": BASE_N,
    "sweeps": {
        name: {
            "label": spec["label"],
            "parameter_name": spec["parameter_name"],
            "n_values": len(spec["values"]),
            "min_value": float(np.min(spec["values"])),
            "max_value": float(np.max(spec["values"])),
        }
        for name, spec in SWEEPS.items()
    },
    "pca_variance_ratio": [float(x) for x in pca.explained_variance_ratio_],
    "figures": [
        "boundary_ws_rewiring_path.png",
        "boundary_ws_family_assignment.png",
        "boundary_modular_bridge_path.png",
        "boundary_modular_family_assignment.png",
        "boundary_degree_heterogeneity_path.png",
        "boundary_degree_family_assignment.png",
        "boundary_chord_density_path.png",
        "boundary_chord_family_assignment.png",
        "boundary_surface_summary.png",
        "boundary_confidence_minima.png",
        "boundary_curvature_diagnostics.png",
    ],
    "results": [
        "boundary_baseline_residual_manifold.csv",
        "boundary_all_sweeps.csv",
        "boundary_ws_rewiring.csv",
        "boundary_modular_bridge.csv",
        "boundary_degree_heterogeneity.csv",
        "boundary_chord_density.csv",
        "boundary_switch_points.csv",
        "boundary_curve_diagnostics.csv",
        "boundary_surface_summary.csv",
        "boundary_detection_summary.json",
    ],
}

summary_path = RESULTS / "boundary_detection_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

md = f'''# Notebook 23 - Universality Boundary Detection

## Core question

Where do graph families stop behaving like one residual universality class and transition into another?

## Core claim

{summary["core_claim"]}

## Boundary sweeps

- Watts-Strogatz rewiring
- SBM modular bridge
- preferential-attachment degree heterogeneity
- cycle chord density

## Main outputs

- `figures/boundary_surface_summary.png`
- `figures/boundary_confidence_minima.png`
- `figures/boundary_curvature_diagnostics.png`
- `results/boundary_switch_points.csv`
- `results/boundary_surface_summary.csv`
'''

md_path = DOCS / "notebook_23_boundary_detection.md"
md_path.write_text(md, encoding="utf-8")

print(json.dumps(summary, indent=2))
print("saved:", summary_path)
print("saved:", md_path)


## 11. Build output archive

In [ ]:

zip_path = ROOT / "notebook_23_outputs.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIGURES, RESULTS, DOCS]:
        if folder.exists():
            for file in folder.glob("*"):
                if file.is_file() and (
                    file.name.startswith("boundary_")
                    or file.name.startswith("notebook_23")
                ):
                    zf.write(file, arcname=f"{folder.name}/{file.name}")

print("saved:", zip_path)

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))



## Interpretation

Notebook 23 shifts the repo from OOD transfer to controlled transition geometry.

Paper-facing interpretation:

- Graph-family continua generate paths through residual manifold coordinates.
- Boundary regions appear as low-confidence intervals between known families.
- Some transitions produce explicit nearest-family switches.
- Other transitions remain inside a family but show high curvature or confidence dips.
- Together, these diagnostics support a residual-geometric view of universality-class transition surfaces.

Recommended next notebook:

**Notebook 24 - Paper Figure Assembly and Results Manifest**

Notebook 24 can gather strongest figures from Notebooks 13-23 into a clean paper-ready figure set.
